# 🎬 Biography Video Factory (v5 – đa ngôn ngữ)
**Cách dùng:** các file `*.json` (Claude tạo) nằm trong `My Drive/DanhNhan/_inbox/` → *Runtime ▸ Run all*.

Kết quả: `My Drive/DanhNhan/output/<tên-nhân-vật>/`
- `images/` — ảnh tư liệu **dùng chung** cho mọi ngôn ngữ (muốn thay ảnh: bỏ `07.jpg` vào đây)
- `Viet/`, `Anh/`, `Phap/`, `TayBanNha/`, `ARap/`, `Trung/` — video, Shorts, thumbnail, mô tả của từng thứ tiếng

Bản Việt nên chạy **trước** các bản dịch (bản dịch tự lấy đúng ảnh bản Việt đã dùng).
Secrets: `VBEE_TOKEN`, `VBEE_APP_ID` (+ `YT_CLIENT_JSON`, `YT_REFRESH_TOKEN…` cho YouTube), bật *Notebook access*.


In [ ]:
# 1) Đăng nhập Google Drive (qua API) + cài font Ả Rập / chữ Hán
from google.colab import auth, userdata
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import io, os, sys, glob, json, shutil, importlib, subprocess

if not os.path.exists('/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc'):
    print('Đang cài font chữ Hán + Ả Rập (chỉ lần đầu mỗi phiên, ~1 phút)…')
    subprocess.run('apt-get -qq update && apt-get -qq install -y fonts-noto-cjk fonts-noto-core libfribidi0 libraqm0',
                   shell=True, capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arabic-reshaper', 'python-bidi'], capture_output=True)

svc = build('drive', 'v3')
FOLDER = 'application/vnd.google-apps.folder'
LOCAL = '/content/factory'; os.makedirs(LOCAL, exist_ok=True)

def q_esc(s): return s.replace("\\", "\\\\").replace("'", "\\'")
def ls(parent, name=None, folder=None):
    q = f"'{parent}' in parents and trashed = false"
    if name: q += f" and name = '{q_esc(name)}'"
    if folder is True:  q += f" and mimeType = '{FOLDER}'"
    if folder is False: q += f" and mimeType != '{FOLDER}'"
    out, tok = [], None
    while True:
        r = svc.files().list(q=q, fields='nextPageToken, files(id,name,mimeType)', pageToken=tok,
                             orderBy='name', pageSize=200).execute()
        out += r['files']; tok = r.get('nextPageToken')
        if not tok: return out
def folder(name, parent):
    f = ls(parent, name, folder=True)
    if f: return f[0]['id']
    return svc.files().create(body={'name': name, 'mimeType': FOLDER, 'parents': [parent]}, fields='id').execute()['id']
def download(fid, path):
    with io.FileIO(path, 'wb') as fh:
        d = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid), chunksize=50 * 1024 * 1024); done = False
        while not done: _, done = d.next_chunk()
def upload(path, parent):
    media = MediaFileUpload(path, resumable=True, chunksize=50 * 1024 * 1024)
    old = ls(parent, os.path.basename(path), folder=False)
    if old: svc.files().update(fileId=old[0]['id'], media_body=media).execute()
    else:   svc.files().create(body={'name': os.path.basename(path), 'parents': [parent]}, media_body=media, fields='id').execute()
def move(fid, src, dst):
    svc.files().update(fileId=fid, addParents=dst, removeParents=src).execute()

# Tìm thư mục DanhNhan có chứa video_pipeline.py
ROOT_ID = None
for f in ls('root', 'DanhNhan', folder=True):
    if ls(f['id'], 'video_pipeline.py', folder=False):
        ROOT_ID = f['id']; break
if not ROOT_ID:
    raise SystemExit('❌ Không thấy My Drive/DanhNhan/video_pipeline.py')
IDS = {s: folder(s, ROOT_ID) for s in ['_inbox', '_done', 'output', 'music']}

download(ls(ROOT_ID, 'video_pipeline.py', folder=False)[0]['id'], f'{LOCAL}/video_pipeline.py')
yt_file = ls(ROOT_ID, 'yt_upload.py', folder=False)
if yt_file:
    download(yt_file[0]['id'], f'{LOCAL}/yt_upload.py')
sys.path.insert(0, LOCAL)
import video_pipeline as vp; importlib.reload(vp)
print('Hỗ trợ chữ Ả Rập:', 'raqm' if vp.RAQM else 'arabic-reshaper')

# Chuyển kết quả cũ (trước v5, nằm thẳng trong output/<tên>/) vào thư mục con Viet/ — chỉ làm một lần
LANG_DIRS = {v['folder'] for v in vp.LANGS.values()}
def migrate(slug_id):
    items = ls(slug_id)
    old = [x for x in items if x['name'] not in LANG_DIRS | {'images', 'image_report.txt'}]
    if not old: return 0
    viet = folder('Viet', slug_id)
    for x in old: move(x['id'], slug_id, viet)
    return len(old)
for f in ls(IDS['output'], folder=True):
    n = migrate(f['id'])
    if n: print(f"↪ {f['name']}: chuyển {n} mục cũ vào {f['name']}/Viet")

JOBS = [f for f in ls(IDS['_inbox'], folder=False) if f['name'].endswith('.json')]
print('✅ Đã nạp pipeline. Gói đang chờ:', [f['name'] for f in JOBS])


In [ ]:
# 2) Cấu hình (chỉ sửa ở đây)
CFG = {
    'VBEE_TOKEN':  userdata.get('VBEE_TOKEN'),
    'VBEE_APP_ID': userdata.get('VBEE_APP_ID'),
    # Mã giọng Vbee cho từng ngôn ngữ — lấy trên trang Vbee (AIVoice API ▸ danh sách giọng / voice code)
    'VOICES': {
        'vi': 'hn_male_phuthang_stor80dt_48k-fhg',
        'en': '',   # tiếng Anh (giọng Mỹ, nam, trầm, kể chuyện)
        'fr': '',   # tiếng Pháp
        'es': '',   # tiếng Tây Ban Nha (châu Mỹ)
        'ar': '',   # tiếng Ả Rập
        'zh': '',   # tiếng Trung phổ thông
    },
    'SPEED':  1.0,                    # tốc độ mặc định (0.25 – 1.9); Shorts tự chậm hơn 5%
    'SPEEDS': {},                     # tốc độ riêng, ví dụ {'ar': 0.95}
    'VBEE_WEBHOOK':'https://example.com/vbee-callback',
    'CONTACT':     '',
    'VOICE_TEST':  False,             # True = ô 2b đọc thử 1 câu bằng mọi giọng đã điền (rất ít ký tự)

    # ---- YouTube: mỗi ngôn ngữ một kênh ----
    'YT_TEST_UPLOAD':  False,
    'YT_AUTO_UPLOAD':  False,         # CHỈ bật sau khi YouTube duyệt dự án API
    'YT_PRIVACY':      'public',
    'YT_AUTH_LANG':    '',            # cấp quyền cho kênh mới: điền 'en' / 'fr' / … rồi chạy ô 3
    'YT_CHANNELS': {   # secret chứa token, playlist, giờ đăng theo múi giờ khán giả
        'vi': dict(secret='YT_REFRESH_TOKEN',    playlist='Danh nhân thế giới',               time='19:00', tz='Asia/Ho_Chi_Minh'),
        'en': dict(secret='YT_REFRESH_TOKEN_EN', playlist='World Famous Figures',             time='19:00', tz='America/New_York'),
        'fr': dict(secret='YT_REFRESH_TOKEN_FR', playlist="Grandes figures de l'histoire",    time='19:00', tz='Europe/Paris'),
        'es': dict(secret='YT_REFRESH_TOKEN_ES', playlist='Grandes personajes de la historia', time='20:00', tz='America/Mexico_City'),
        'ar': dict(secret='YT_REFRESH_TOKEN_AR', playlist='عظماء العالم',                      time='20:00', tz='Africa/Cairo'),
        'zh': dict(secret='YT_REFRESH_TOKEN_ZH', playlist='世界名人傳記',                       time='20:00', tz='Asia/Taipei'),
    },
}
mp3s = [f for f in ls(IDS['music'], folder=False) if f['name'].lower().endswith('.mp3')]
MUSIC = None
if mp3s:
    MUSIC = f"{LOCAL}/music.mp3"; download(mp3s[0]['id'], MUSIC)
print('Nhạc nền:', mp3s[0]['name'] if mp3s else 'không có')
print('Giọng đã điền:', [k for k, v in CFG['VOICES'].items() if v])


In [ ]:
# 2b) (Tuỳ chọn) Nghe thử giọng của từng ngôn ngữ — bật CFG['VOICE_TEST'] = True
from pathlib import Path
from IPython.display import Audio, display
SAMPLES = {
    'vi': 'Năm mười chín tuổi, ông vẫn chỉ là một cậu sinh viên nghèo phải làm thuê để được đi học.',
    'en': 'At nineteen, he was still a poor student who had to work as a servant just to stay in school.',
    'fr': "À dix-neuf ans, il n'était encore qu'un étudiant pauvre, obligé de servir pour pouvoir étudier.",
    'es': 'A los diecinueve años, todavía era un estudiante pobre que trabajaba como sirviente para poder estudiar.',
    'ar': 'في التاسعة عشرة من عمره، كان لا يزال طالبًا فقيرًا يعمل خادمًا كي يتمكن من الدراسة.',
    'zh': '十九歲那年，他還只是一個貧窮的學生，必須當僕人才能繼續讀書。',
}
if CFG.get('VOICE_TEST'):
    os.makedirs(f'{LOCAL}/voice_test', exist_ok=True)
    for lang, code in CFG['VOICES'].items():
        if not code: continue
        out = Path(f'{LOCAL}/voice_test/{lang}.mp3')
        vp.vbee_tts(SAMPLES[lang], out, vp.voice_cfg(CFG, lang))
        print(lang, code); display(Audio(str(out)))
else:
    print('Bỏ qua nghe thử (VOICE_TEST = False).')


In [ ]:
# 3) Kết nối YouTube — mỗi ngôn ngữ một kênh (cấp quyền mỗi kênh một lần)
def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

YTS = {}
if os.path.exists(f'{LOCAL}/yt_upload.py'):
    import yt_upload as ytu; importlib.reload(ytu)
    from google.colab import output
    cj = secret('YT_CLIENT_JSON')
    if not cj:
        print('ℹ️ Chưa có YT_CLIENT_JSON trong Secrets → bỏ qua phần YouTube.')
    else:
        want = CFG.get('YT_AUTH_LANG', '')
        if want:
            name = CFG['YT_CHANNELS'][want]['secret']
            print(f'Cấp quyền cho kênh ngôn ngữ "{want}". Khi Google hỏi, CHỌN ĐÚNG KÊNH của ngôn ngữ này.\n')
            rt = ytu.authorize_once(cj)
            print(f'\n③ REFRESH TOKEN (bí mật như mật khẩu):\n\n{rt}')
            print(f'\n④ 🔑 Secrets → Add new secret → Name: {name} → Value: dán token → bật Notebook access.')
            print("⑤ Sau đó sửa CFG['YT_AUTH_LANG'] = '' ở ô 2.")
            input('\nChép xong thì bấm Enter để XÓA token khỏi màn hình… ')
            output.clear(); print('✅ Đã xóa token khỏi màn hình.')
        for lang, ch in CFG['YT_CHANNELS'].items():
            rt = secret(ch['secret'])
            if rt:
                YTS[lang] = ytu.service(cj, rt)
        print('✅ Kênh đã kết nối:', list(YTS) or 'chưa có')
else:
    print('ℹ️ Chưa có yt_upload.py → bỏ qua phần YouTube.')
YT = YTS.get('vi')


In [ ]:
# 4) (Tuỳ chọn) Tải thử một clip test 15 giây — làm bằng chứng cho đơn kiểm duyệt YouTube
from pathlib import Path
if CFG.get('YT_TEST_UPLOAD') and YT:
    from PIL import Image, ImageDraw
    img = Image.new('RGB', (1920, 1080), (18, 22, 32)); d = ImageDraw.Draw(img)
    d.text((140, 420), 'DanhNhan Video Uploader', font=vp.font(96), fill=(255, 255, 255))
    d.text((140, 560), 'YouTube Data API upload test', font=vp.font(60, bold=False), fill=(232, 180, 60))
    img.save(f'{LOCAL}/api_test.jpg', quality=92)
    vp.run(['ffmpeg', '-y', '-loop', '1', '-i', f'{LOCAL}/api_test.jpg', '-f', 'lavfi', '-i', 'anullsrc=r=48000:cl=stereo',
            '-t', '15', '-r', '25', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-shortest', f'{LOCAL}/api_test.mp4'])
    body = ytu.build_body('DanhNhan Video Uploader – API upload test',
                          'Test upload made with DanhNhan Video Uploader (YouTube Data API v3).', ['test'], 'private')
    vid = ytu.upload(YT, Path(f'{LOCAL}/api_test.mp4'), body, Path(f'{LOCAL}/api_test.jpg'), vp.log)
    print(f'👉 Mở YouTube Studio để xem video test: https://studio.youtube.com/video/{vid}/edit')
else:
    print('Bỏ qua tải thử (YT_TEST_UPLOAD = False).')


In [ ]:
# 5) Sản xuất mọi gói trong _inbox → output/<tên>/<Ngôn ngữ>/ (và YouTube nếu đã bật)
from pathlib import Path
def pull(drive_parent, local_dir, overwrite=True):
    os.makedirs(local_dir, exist_ok=True)
    for a in ls(drive_parent, folder=False):
        p = f"{local_dir}/{a['name']}"
        if overwrite or not os.path.exists(p):
            download(a['id'], p)

def push_results(root, out, slug, lang_dir):
    slug_id = folder(slug, IDS['output']); dst = folder(lang_dir, slug_id)
    for p in sorted(glob.glob(f'{out}/*')):                      # video, Shorts, thumbnail, mô tả…
        if os.path.isfile(p): upload(p, dst)
    for sub, src in [('slides', f'{out}/slides'), ('audio', f'{out}/_work/audio')]:
        files = sorted(glob.glob(f'{src}/*'))
        if files:
            sid = folder(sub, dst)
            for p in files: upload(p, sid)
    files = sorted(glob.glob(f'{root}/images/*'))                  # ảnh dùng chung
    if files:
        iid = folder('images', slug_id)
        for p in files: upload(p, iid)
    if os.path.exists(f'{root}/image_report.txt'): upload(f'{root}/image_report.txt', slug_id)
    return dst

# bản Việt chạy trước, bản dịch chạy sau (để dùng đúng ảnh bản Việt)
pk = []
for job in JOBS:
    path = f"{LOCAL}/inbox/{job['name']}"; os.makedirs(os.path.dirname(path), exist_ok=True)
    download(job['id'], path)
    pk.append((job, path, json.load(open(path, encoding='utf-8'))))
pk.sort(key=lambda x: vp.norm_lang(x[2].get('lang')) != 'vi')

n_by_lang = {}
for job, path, pkg in pk:
    slug, lang = pkg['slug'], vp.norm_lang(pkg.get('lang'))
    lang_dir = vp.LANGS[lang]['folder']
    root, out = f'{LOCAL}/output/{slug}', f'{LOCAL}/output/{slug}/{lang_dir}'
    print(f'\n━━━━ {job["name"]}  →  output/{slug}/{lang_dir}')
    slug_id = folder(slug, IDS['output']); migrate(slug_id)
    # lấy lại ảnh dùng chung + audio đã có (không tốn Vbee lần 2) + bản Việt để đồng bộ ảnh
    old = ls(slug_id, 'images', folder=True)
    if old: pull(old[0]['id'], f'{root}/images')
    ldir = ls(slug_id, lang_dir, folder=True)
    if ldir:
        a = ls(ldir[0]['id'], 'audio', folder=True)
        if a: pull(a[0]['id'], f'{out}/_work/audio', overwrite=False)
    if lang != 'vi':
        v = ls(slug_id, 'Viet', folder=True)
        pj = ls(v[0]['id'], 'production.json', folder=False) if v else []
        if pj:
            os.makedirs(f'{root}/Viet', exist_ok=True); download(pj[0]['id'], f'{root}/Viet/production.json')
        else:
            print('   ⚠ chưa có bản Việt trên Drive — bản dịch dùng ảnh theo link trong JSON')
    try:
        vp.build(path, f'{LOCAL}/output', CFG, music_path=MUSIC)
        vp.log('Đang tải kết quả lên Google Drive…')
        dst = push_results(root, out, slug, lang_dir)
        yt = YTS.get(lang)
        if CFG.get('YT_AUTO_UPLOAD'):
            if not yt:
                vp.log(f'  ⚠ chưa kết nối kênh YouTube cho "{lang}" → không tải lên tự động')
            else:
                ch = CFG['YT_CHANNELS'][lang]; n = n_by_lang.get(lang, 0); n_by_lang[lang] = n + 1
                pub = ytu.next_publish_time(ch['time'], ch['tz'], days_ahead=n) if ch.get('time') else None
                vp.log(f'Đang tải lên kênh YouTube "{lang}"…' + (f' (hẹn đăng {pub} UTC)' if pub else ''))
                ytu.upload_folder(yt, Path(out), privacy=CFG.get('YT_PRIVACY', 'public'), publish_at=pub,
                                  log=vp.log, playlist=ch.get('playlist'))
                if os.path.exists(f'{out}/youtube_video_id.txt'): upload(f'{out}/youtube_video_id.txt', dst)
        move(job['id'], IDS['_inbox'], IDS['_done'])
        print(f'🎉 XONG: https://drive.google.com/drive/folders/{dst}')
    except BaseException as e:
        print('❌', job['name'], '→', e)
        try:
            push_results(root, out, slug, lang_dir); print('   (đã lưu phần làm được lên Drive, chạy lại không tốn Vbee lần 2)')
        except Exception:
            pass
        if isinstance(e, KeyboardInterrupt): raise
